# Paper-Accurate Hashimoto Algorithm 1 + DP-SGD + Privacy Auditing
Merged from uploaded files.

Notes from quick inspection:
- The code appears truncated in the first file and continued in the second file.
- `audit_epsilon()` contains a likely bug: `max((scores > tau).astype(int) == m for tau in thresholds).sum()` should probably compute the maximum count across thresholds, e.g. `max((((scores > tau).astype(int) == m).sum()) for tau in thresholds)`.
- `train_model()` rebuilds `train_loader` after DRO updates, but the new loader is not re-wrapped by Opacus' privacy engine. Verify privacy accounting remains correct.
- Verify `model._module` exists in your installed Opacus version before loading state dicts.


In [1]:
"""
Paper-Accurate Hashimoto et al. (2018) Algorithm 1 + DP-SGD + Steinke Audit
===========================================================================
Run using: CUDA_VISIBLE_DEVICES=1 jupyter notebook

Key Implementation Details:
- EXACT Algorithm 1: Maintains q_t distribution and lambda_t dual variable.
- DRO Resampling: Applies q_t via WeightedRandomSampler so DP-SGD gradient 
  clipping does not destroy the minority upweighting signal.
- WB Bypass: Copies weights to a fresh vanilla model to avoid Opacus crashes.
"""

import os, sys, json, warnings, gc, time, random, math
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Tuple, Optional
from collections import defaultdict

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import stats
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split, Subset, ConcatDataset, WeightedRandomSampler
from torchvision import transforms

from opacus import PrivacyEngine
from opacus.validators import ModuleValidator
from opacus.utils.batch_memory_manager import BatchMemoryManager

warnings.filterwarnings('ignore')

In [2]:
# ══════════════════════════════════════════════════════════════════════════════
# FIX: Patch BatchMemoryManager for Opacus compatibility
# ══════════════════════════════════════════════════════════════════════════════
def _patched_batch_memory_iter(self):
    for batch in self.data_loader:
        for i in range(0, len(batch[0]), self.max_physical_batch_size):
            yield tuple(x[i : i + self.max_physical_batch_size] for x in batch)

BatchMemoryManager.__iter__ = _patched_batch_memory_iter

In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# DEVICE & CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} (If using CUDA_VISIBLE_DEVICES=1, this is your 2nd L40s)')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} - {props.total_memory / 1e9:.1f} GB')

RACE_MAP = {0: 'White', 1: 'Black', 2: 'Asian', 3: 'Indian', 4: 'Others'}

CFG = {
    'image_dir':  Path('project/data/UTKFace'),
    'exp_dir':    Path('project/experiments/paper_accurate_alg1'),
    
    'epochs':       25,
    'lr':           1e-4,
    'batch_size':   64,
    'weight_decay': 1e-4,
    'val_frac':     0.15,
    'seed':         2024,

    # ── Hashimoto et al. Algorithm 1 Parameters ─────────────────────────
    # 'erm': Standard baseline
    # 'alg1_chisq': Exact Algorithm 1 with chi-squared mirror descent
    'dro_mode':            'alg1_chisq', 
    
    'dro_alpha':           1.0,      # chi-squared ball radius (alpha in paper)
    'dro_eta':             0.01,     # Step size for lambda update (eta in paper)
    'dro_warmup_epochs':   1,        # Epochs of standard ERM before DRO kicks in

    # ── DP-SGD ──────────────────────────────────────────────────────────
    'dp_noise_multiplier': 1.1,
    'dp_max_grad_norm':    1.0,
    'dp_delta':            1e-5,
    'dp_phys_batch':       32,

    # ── Privacy Auditing ────────────────────────────────────────────────
    'audit_k':             500,
    'audit_gamma':         0.05,
    'audit_seed':          42,
    'wb_score_epochs':     5,
    'wb_sample_batches':   10,
}

CFG['exp_dir'].mkdir(parents=True, exist_ok=True)
random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG['seed'])

Device: cuda (If using CUDA_VISIBLE_DEVICES=1, this is your 2nd L40s)
GPU: NVIDIA L40S - 47.7 GB


In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET & MODEL (Unchanged)
# ══════════════════════════════════════════════════════════════════════════════
class UTKFaceDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir, self.transform, self.samples = Path(root_dir), transform, []
        for p in sorted(self.root_dir.rglob('*.jpg')):
            try:
                parts = p.stem.replace('.chip', '').split('_')
                if len(parts) >= 4:
                    age, gender, race = int(parts[0]), int(parts[1]), int(parts[2])
                    if 1 <= age <= 116 and gender in (0, 1) and race in RACE_MAP:
                        self.samples.append({'path': str(p), 'age': age, 'gender': gender, 'race': race})
            except:
                pass

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(s['age'], dtype=torch.float32), torch.tensor(s['gender'], dtype=torch.long), torch.tensor(s['race'], dtype=torch.long)

    def get_pil_image(self, idx):
        return Image.open(self.samples[idx]['path']).convert('RGB')

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.ColorJitter(0.2, 0.2, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

canary_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Loading data...')
full_ds = UTKFaceDataset(CFG['image_dir'], transform=None)
N = len(full_ds)
k = CFG['audit_k']
audit_rng = np.random.default_rng(CFG['audit_seed'])

all_indices = list(range(N))
audit_rng.shuffle(all_indices)
canary_in_pool, canary_out_pool = all_indices[:k], all_indices[k:2*k]
remaining_indices = all_indices[2*k:]
n_val = int(len(remaining_indices) * CFG['val_frac'])

gen = torch.Generator().manual_seed(CFG['seed'])
train_idx, val_idx = random_split(remaining_indices, [len(remaining_indices)-n_val, n_val], generator=gen)
train_idx, val_idx = list(train_idx), list(val_idx)

class TransformSubset(Dataset):
    def __init__(self, base, indices, transform):
        self.base, self.indices, self.tf = base, indices, transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, age, g, r = self.base[self.indices[i]]
        return self.tf(img), age, g, r

train_ds = TransformSubset(full_ds, train_idx, train_tf)
val_ds = TransformSubset(full_ds, val_idx, val_tf)

class GNBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv2d(inplanes, planes, 3, stride=stride, padding=1, bias=False)
        self.gn1 = nn.GroupNorm(min(32, planes), planes)
        self.relu = nn.ReLU(inplace=False)
        self.conv2 = nn.Conv2d(planes, planes, 3, padding=1, bias=False)
        self.gn2 = nn.GroupNorm(min(32, planes), planes)
        self.downsample = downsample

    def forward(self, x):
        out = self.relu(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        if self.downsample:
            x = self.downsample(x)
        return self.relu(out + x)

class OpacusResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.inplanes = 64
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.gn1 = nn.GroupNorm(min(32, 64), 64)
        self.relu = nn.ReLU(inplace=False)
        self.maxpool = nn.MaxPool2d(3, stride=2, padding=1)
        self.layer1 = self._make_layer(64, 2, 1)
        self.layer2 = self._make_layer(128, 2, 2)
        self.layer3 = self._make_layer(256, 2, 2)
        self.layer4 = self._make_layer(512, 2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(512, 1)

    def _make_layer(self, planes, blocks, stride):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            downsample = nn.Sequential(
                nn.Conv2d(self.inplanes, planes, 1, stride=stride, bias=False),
                nn.GroupNorm(min(32, planes), planes)
            )
        layers = [GNBlock(self.inplanes, planes, stride, downsample)]
        self.inplanes = planes
        for _ in range(1, blocks):
            layers.append(GNBlock(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.maxpool(self.relu(self.gn1(self.conv1(x))))
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.fc(torch.flatten(self.avgpool(x), 1)).squeeze(1)

def build_model():
    m = OpacusResNet18()
    if ModuleValidator.validate(m, strict=False):
        m = ModuleValidator.fix(m)
    return m.to(device)

Loading data...


In [5]:
# At the top of your notebook, after RACE_MAP definition
RACE_NAMES = ['white', 'black', 'asian', 'indian', 'others']

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# CANARY DATASET
# ══════════════════════════════════════════════════════════════════════════════
class CanaryDataset:
    def __init__(self, base_ds, in_idx, out_idx, membership, transform, rng):
        self.base_ds = base_ds
        self.membership = membership
        self.k = len(membership)
        self.image_indices = [in_idx[i] if m else out_idx[i] for i, m in enumerate(membership)]
        held_out_ages = [base_ds.samples[i]['age'] for i in val_idx]
        rng.shuffle(held_out_ages)
        self.shuffled_ages = torch.tensor(held_out_ages[:self.k], dtype=torch.float32)
        print(f'Pre-computing {self.k} canary images...')
        self.canary_images = [transform(base_ds.get_pil_image(idx)) for idx in self.image_indices]

    def get_in_dataset(self):
        in_idxs = [i for i, m in enumerate(self.membership) if m]
        return _InCanarySubset(self, in_idxs), in_idxs

    def get_all_canary_batch(self):
        return torch.stack(self.canary_images).to(device), self.shuffled_ages.to(device)

class _InCanarySubset(Dataset):
    def __init__(self, cd, in_idxs):
        self.cd = cd
        self.in_idxs = in_idxs

    def __len__(self):
        return len(self.in_idxs)

    def __getitem__(self, i):
        img = self.cd.canary_images[self.in_idxs[i]].clone()
        age = self.cd.shuffled_ages[self.in_idxs[i]]
        return img, age, torch.tensor(-1, dtype=torch.long), torch.tensor(-1, dtype=torch.long)

membership = torch.tensor(audit_rng.integers(0, 2, size=k).astype(bool), dtype=torch.bool)
canary_ds = CanaryDataset(full_ds, canary_in_pool, canary_out_pool, membership, canary_tf, audit_rng)
in_canary_ds, in_canary_global_idxs = canary_ds.get_in_dataset()
augmented_ds = ConcatDataset([train_ds, in_canary_ds])

Pre-computing 500 canary images...


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION & AUDIT UTILITIES (FIXED)
# ══════════════════════════════════════════════════════════════════════════════

RACE_NAMES = ['white', 'black', 'asian', 'indian', 'others']
RACE_MAP = {0: 'White', 1: 'Black', 2: 'Asian', 3: 'Indian', 4: 'Others'}

def evaluate_fairness(model, loader):
    model.eval()
    gender_loss = {0: 0.0, 1: 0.0}
    gender_count = {0: 0, 1: 0}
    race_loss = {r: 0.0 for r in RACE_MAP}
    race_count = {r: 0 for r in RACE_MAP}
    total_loss = 0.0
    total_count = 0
    
    with torch.no_grad():
        for imgs, ages, genders, races in loader:
            imgs, ages = imgs.to(device), ages.to(device)
            abs_err = (model(imgs) - ages).abs().cpu()
            valid = (genders >= 0)
            
            total_loss += abs_err[valid].sum().item()
            total_count += valid.sum().item()
            
            for g in (0, 1):
                mask = (genders == g) & valid
                gender_loss[g] += abs_err[mask].sum().item()
                gender_count[g] += mask.sum().item()
            
            for r in RACE_MAP:
                mask = (races == r) & valid
                race_loss[r] += abs_err[mask].sum().item()
                race_count[r] += mask.sum().item()
    
    res = {
        'overall_mae': total_loss / max(total_count, 1),
        'male_mae': gender_loss[0] / max(gender_count[0], 1),
        'female_mae': gender_loss[1] / max(gender_count[1], 1),
        'gender_gap': abs(gender_loss[0] / max(gender_count[0], 1) - gender_loss[1] / max(gender_count[1], 1))
    }
    
    # FIX: Use RACE_NAMES (strings) instead of RACE_MAP (ints)
    for race_name in RACE_NAMES:
        # Find the race ID that corresponds to this name
        race_id = [k for k, v in RACE_MAP.items() if v.lower() == race_name][0]
        res[f'{race_name}_mae'] = race_loss[race_id] / max(race_count[race_id], 1)
    
    race_maes = [res[f'{r}_mae'] for r in RACE_NAMES]
    res['race_gap'] = max(race_maes) - min(race_maes)
    
    return res


def audit_epsilon(scores, membership, gamma=0.05):
    m = membership.numpy().astype(int)
    k = len(scores)
    thresholds = np.linspace(scores.min() - 1e-9, scores.max() + 1e-9, 1000)
    
    best_V = 0
    best_tau = thresholds[0]
    for tau in thresholds:
        guesses = (scores > tau).astype(int)
        V = (guesses == m).sum()
        if V > best_V:
            best_V = V
            best_tau = tau
    
    p_null = stats.binom.sf(best_V - 1, k, 0.5)
    
    def p_val(eps):
        p_succ = math.exp(eps) / (1 + math.exp(eps))
        return stats.binom.sf(best_V - 1, k, p_succ)
    
    lo, hi = 0.0, 20.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if p_val(mid) > gamma:
            lo = mid
        else:
            hi = mid
    
    in_s = scores[m == 1]
    out_s = scores[m == 0]
    pooled_std = np.sqrt((in_s.var() + out_s.var()) / 2)
    
    try:
        from sklearn.metrics import roc_auc_score
        auc = roc_auc_score(m, scores)
    except:
        auc = 0.5
    
    return {
        'eps_lb': lo if p_val(lo) <= gamma else 0.0,
        'V': int(best_V),
        'k': k,
        'threshold': best_tau,
        'p_value': float(p_null),
        'attack_acc': best_V / k,
        'delta': in_s.mean() - out_s.mean(),
        'cohens_d': (in_s.mean() - out_s.mean()) / (pooled_std + 1e-8),
        'auc': auc
    }

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING FUNCTION (EXACT ALGORITHM 1 + DP-SGD) - FIXED
# ══════════════════════════════════════════════════════════════════════════════

RACE_NAMES = ['white', 'black', 'asian', 'indian', 'others']

def train_model(cfg, train_ds, val_loader, canary_ds, membership, experiment_name='exp'):
    n_train = len(train_ds)
    k = len(membership)
    alpha = cfg['dro_alpha']
    eta = cfg['dro_eta']
    is_alg1 = (cfg['dro_mode'] == 'alg1_chisq')
    
    # Initialize Algorithm 1 variables
    q_t = np.ones(n_train) / n_train
    lambda_t = 0.0
    
    # Initial uniform DataLoader
    init_loader = DataLoader(augmented_ds, batch_size=cfg['batch_size'], shuffle=True, num_workers=4, pin_memory=True)
    
    model = build_model()
    optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    criterion = nn.L1Loss(reduction='mean')
    
    privacy_engine = PrivacyEngine()
    model, optimizer, train_loader = privacy_engine.make_private(
        module=model, 
        optimizer=optimizer, 
        data_loader=init_loader,
        noise_multiplier=cfg['dp_noise_multiplier'], 
        max_grad_norm=cfg['dp_max_grad_norm']
    )
    
    # Compute initial canary losses
    canary_imgs, canary_ages = canary_ds.get_all_canary_batch()
    with torch.no_grad():
        initial_losses = (model(canary_imgs.to(device)) - canary_ages.to(device)).abs().cpu().numpy()
    
    cumulative_delta_w = None
    wb_scores = np.zeros(k, dtype=np.float64)
    
    history = {key: [] for key in ['train_loss', 'overall_mae', 'male_mae', 'female_mae', 'gender_gap', 'race_gap', 'epsilon', 'lambda_t', 'target_var']}
    # FIX: Use RACE_NAMES instead of RACE_MAP
    for r in RACE_NAMES:
        history[f'{r}_mae'] = []
    
    print(f'[{experiment_name}] Training ({cfg["epochs"]} epochs, mode={cfg["dro_mode"]})...')
    start_time = time.time()
    
    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        epoch_loss = 0.0
        epoch_n = 0
        do_wb = (epoch % cfg['wb_score_epochs'] == 0 or epoch == cfg['epochs'])
        
        if do_wb:
            total_batches = len(train_loader)
            wb_batches = set(int(x) for x in np.random.choice(
                total_batches, 
                min(cfg['wb_sample_batches'], total_batches), 
                replace=False
            ))
            epoch_delta_w = None
        
        with BatchMemoryManager(
            data_loader=train_loader, 
            max_physical_batch_size=cfg['dp_phys_batch'], 
            optimizer=optimizer
        ) as loader_iter:
            for batch_idx, (imgs, ages, _, _) in enumerate(loader_iter):
                imgs, ages = imgs.to(device), ages.to(device)
                
                params_before = None
                if do_wb and int(batch_idx) in wb_batches:
                    params_before = [p.detach().clone() for p in model.parameters() if p.requires_grad]
                
                loss = criterion(model(imgs), ages)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item() * imgs.size(0)
                epoch_n += imgs.size(0)
                
                if params_before is not None:
                    batch_delta = torch.cat([
                        (b - a).view(-1) 
                        for b, a in zip(
                            params_before, 
                            [p.detach() for p in model.parameters() if p.requires_grad]
                        )
                    ])
                    epoch_delta_w = batch_delta if epoch_delta_w is None else epoch_delta_w + batch_delta

        # --- END OF EPOCH: HASHIMOTO ALGORITHM 1 UPDATE ---
        if is_alg1 and epoch >= cfg['dro_warmup_epochs']:
            model.eval()
            
            eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False, num_workers=2)
            train_losses = []
            with torch.no_grad():
                for im, ag, _, _ in eval_loader:
                    train_losses.extend((model(im.to(device)) - ag.to(device)).abs().cpu().numpy())
            train_losses = np.array(train_losses)
            
            mu_t = np.sum(q_t * train_losses)
            V_t = np.sum(q_t * (train_losses - mu_t)**2)
            
            lambda_t = np.clip(lambda_t + eta * (alpha - V_t), 0.0, 1.0 / (2.0 * alpha))
            
            q_new = q_t * (1.0 + lambda_t * (train_losses - mu_t))
            q_new = np.maximum(q_new, 0.0)
            q_t = q_new / q_new.sum()
            
            canary_q = np.ones(len(in_canary_ds)) / len(in_canary_ds)
            all_weights = np.concatenate([q_t, canary_q])
            new_sampler = WeightedRandomSampler(
                weights=torch.from_numpy(all_weights).double(), 
                num_samples=len(all_weights), 
                replacement=True
            )
            train_loader = DataLoader(
                augmented_ds, 
                batch_size=cfg['batch_size'], 
                sampler=new_sampler, 
                num_workers=4, 
                pin_memory=True
            )
            
            if epoch % 5 == 0:
                print(f'  [DRO Update] λ={lambda_t:.4f}, Var={V_t:.2f}, Target α={alpha:.2f}')

        # --- WHITE BOX SCORING (Safe Bypass) ---
        if do_wb and epoch_delta_w is not None:
            cumulative_delta_w = epoch_delta_w if cumulative_delta_w is None else cumulative_delta_w + epoch_delta_w
            
            wb_model = build_model()
            wb_model.load_state_dict(model._module.state_dict())
            wb_model.eval()
            
            canary_imgs, canary_ages = canary_ds.get_all_canary_batch()
            wb_cont = np.zeros(k, dtype=np.float64)
            
            for s in range(0, k, 64):
                e = min(s + 64, k)
                sub_imgs = canary_imgs[s:e].clone().detach().requires_grad_(True)
                sub_ages = canary_ages[s:e]
                
                sub_grads = torch.autograd.grad(
                    (wb_model(sub_imgs) - sub_ages).abs().sum(), 
                    list(wb_model.parameters())
                )
                
                grad_flat = torch.cat([g.view(-1) for g in sub_grads])
                clip_f = min(1.0, cfg['dp_max_grad_norm'] / (grad_flat.norm().item() + 1e-8))
                wb_cont[s:e] = torch.dot(cumulative_delta_w, grad_flat * clip_f).item() / len(wb_batches)
            
            wb_scores = wb_cont
            del wb_model
            torch.cuda.empty_cache()

        # --- Evaluation ---
        fair = evaluate_fairness(model, val_loader)
        
        for key in ['overall_mae', 'male_mae', 'female_mae', 'gender_gap', 'race_gap']:
            history[key].append(fair[key])
        
        # FIX: Use RACE_NAMES instead of RACE_MAP
        for r in RACE_NAMES:
            history[f'{r}_mae'].append(fair[f'{r}_mae'])
        
        history['train_loss'].append(epoch_loss / epoch_n)
        history['epsilon'].append(privacy_engine.get_epsilon(cfg['dp_delta']))
        history['lambda_t'].append(lambda_t)
        history['target_var'].append(alpha)
        
        if epoch % 5 == 0:
            elapsed = time.time() - start_time
            print(f'[{experiment_name}] Ep {epoch:2d} | L={epoch_loss/epoch_n:.3f} | '
                  f'M={fair["male_mae"]:.2f} F={fair["female_mae"]:.2f} '
                  f'Gap={fair["gender_gap"]:.2f} | ε={history["epsilon"][-1]:.2f} | {elapsed:.0f}s')
    
    # Final canary losses for black-box scores
    canary_imgs, canary_ages = canary_ds.get_all_canary_batch()
    with torch.no_grad():
        final_losses = (model(canary_imgs.to(device)) - canary_ages.to(device)).abs().cpu().numpy()
    
    print(f'[{experiment_name}] Done in {time.time()-start_time:.0f}s')
    
    return {
        'model': model,
        'bb_scores': initial_losses - final_losses,
        'wb_scores': wb_scores,
        'history': history,
        'epsilon_spent': history['epsilon'][-1],
        'final_metrics': fair
    }

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# FIXED TRAINING FUNCTION (DRO SUB-EPOCH SAMPLING)
# ══════════════════════════════════════════════════════════════════════════════

RACE_NAMES = ['white', 'black', 'asian', 'indian', 'others']

def train_model_paper_accurate_fixed(cfg, train_ds, val_loader, canary_ds, membership, experiment_name='exp'):
    """
    FIXED Algorithm 1: Uses DRO Sub-epochs to prevent Opacus from scaling
    noise based on the skewed q_t distribution.
    """
    n_train = len(train_ds)
    k = len(membership)
    alpha = cfg['dro_alpha']
    eta = cfg['dro_eta']
    is_alg1 = (cfg['dro_mode'] == 'alg1_chisq')
    
    # Initialize Algorithm 1 variables
    q_t = np.ones(n_train) / n_train
    lambda_t = 0.0
    
    # Standard uniform loader (Keeps DP noise scaled correctly!)
    def get_uniform_loader(dataset):
        return DataLoader(dataset, batch_size=cfg['batch_size'], shuffle=True, num_workers=4, pin_memory=True)
    
    train_loader = get_uniform_loader(augmented_ds)
    
    model = build_model()
    optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    criterion = nn.L1Loss(reduction='mean')
    
    privacy_engine = PrivacyEngine()
    model, optimizer, train_loader = privacy_engine.make_private(
        module=model, optimizer=optimizer, data_loader=train_loader,
        noise_multiplier=cfg['dp_noise_multiplier'], max_grad_norm=cfg['dp_max_grad_norm'])
    
    # Initial canary losses
    with torch.no_grad():
        c_imgs, c_ages = canary_ds.get_all_canary_batch()
        initial_losses = (model(c_imgs.to(device)) - c_ages.to(device)).abs().cpu().numpy()
    
    cumulative_delta_w = None
    wb_scores = np.zeros(k, dtype=np.float64)
    
    history = {key: [] for key in ['train_loss', 'overall_mae', 'male_mae', 'female_mae', 'gender_gap', 'race_gap', 'epsilon', 'lambda_t', 'target_var']}
    # FIX: Use RACE_NAMES instead of RACE_MAP
    for r in RACE_NAMES:
        history[f'{r}_mae'] = []
    
    print(f'[{experiment_name}] Training ({cfg["epochs"]} epochs, mode={cfg["dro_mode"]})...')
    start_time = time.time()
    
    for epoch in range(1, cfg['epochs'] + 1):
        
        # --- FIX: DRO SUB-EPOCH SAMPLING ---
        if is_alg1 and epoch > cfg['dro_warmup_epochs']:
            # Draw a new subset of training data proportional to q_t
            n_samples = int(n_train * 0.8)
            sampled_train_indices = np.random.choice(n_train, size=n_samples, p=q_t, replace=True)
            # Remove exact duplicates for clean batching (preserve order)
            sampled_train_indices = list(dict.fromkeys(sampled_train_indices))
            
            # Create a temporary dataset for this epoch
            temp_train_ds = TransformSubset(full_ds, [train_idx[i] for i in sampled_train_indices], train_tf)
            epoch_augmented_ds = ConcatDataset([temp_train_ds, in_canary_ds])
            
            # CRITICAL: Wrap in a standard shuffled loader so Opacus assumes uniform probability
            train_loader = get_uniform_loader(epoch_augmented_ds)
        else:
            train_loader = get_uniform_loader(augmented_ds)
        
        model.train()
        epoch_loss = 0.0
        epoch_n = 0
        do_wb = (epoch % cfg['wb_score_epochs'] == 0 or epoch == cfg['epochs'])
        
        if do_wb:
            total_batches = len(train_loader)
            wb_batches = set(int(x) for x in np.random.choice(
                total_batches, 
                min(cfg['wb_sample_batches'], total_batches), 
                replace=False
            ))
            epoch_delta_w = None
        
        with BatchMemoryManager(
            data_loader=train_loader, 
            max_physical_batch_size=cfg['dp_phys_batch'], 
            optimizer=optimizer
        ) as loader_iter:
            for batch_idx, (imgs, ages, _, _) in enumerate(loader_iter):
                imgs, ages = imgs.to(device), ages.to(device)
                
                params_before = None
                if do_wb and int(batch_idx) in wb_batches:
                    params_before = [p.detach().clone() for p in model.parameters() if p.requires_grad]
                
                loss = criterion(model(imgs), ages)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item() * imgs.size(0)
                epoch_n += imgs.size(0)
                
                if params_before is not None:
                    batch_delta = torch.cat([
                        (b - a).view(-1) 
                        for b, a in zip(
                            params_before, 
                            [p.detach() for p in model.parameters() if p.requires_grad]
                        )
                    ])
                    epoch_delta_w = batch_delta if epoch_delta_w is None else epoch_delta_w + batch_delta

        # --- END OF EPOCH: HASHIMOTO ALGORITHM 1 UPDATE ---
        if is_alg1 and epoch >= cfg['dro_warmup_epochs']:
            model.eval()
            # Evaluate on the FULL training set to get true losses
            eval_loader = DataLoader(train_ds, batch_size=256, shuffle=False, num_workers=2)
            train_losses = []
            with torch.no_grad():
                for im, ag, _, _ in eval_loader:
                    train_losses.extend((model(im.to(device)) - ag.to(device)).abs().cpu().numpy())
            train_losses = np.array(train_losses)
            
            # Exact Algorithm 1 Math
            mu_t = np.sum(q_t * train_losses)
            V_t = np.sum(q_t * (train_losses - mu_t)**2)
            
            # Update lambda (Lagrangian dual variable)
            lambda_t = np.clip(lambda_t + eta * (alpha - V_t), 0.0, 1.0 / (2.0 * alpha))
            
            # Update q_t distribution
            q_new = q_t * (1.0 + lambda_t * (train_losses - mu_t))
            q_new = np.maximum(q_new, 1e-12)  # FIX: Prevent zero weights
            q_t = q_new / q_new.sum()
            
            if epoch % 5 == 0:
                print(f'  [DRO Update] λ={lambda_t:.4f}, Var={V_t:.2f}, Target α={alpha:.2f}, q_t range=[{q_t.min():.6f}, {q_t.max():.6f}]')

        # --- WHITE BOX SCORING (Safe Bypass) ---
        if do_wb and epoch_delta_w is not None:
            cumulative_delta_w = epoch_delta_w if cumulative_delta_w is None else cumulative_delta_w + epoch_delta_w
            
            wb_model = build_model()
            wb_model.load_state_dict(model._module.state_dict())
            wb_model.eval()
            
            c_imgs, c_ages = canary_ds.get_all_canary_batch()
            wb_cont = np.zeros(k, dtype=np.float64)
            
            for s in range(0, k, 64):
                e = min(s + 64, k)
                sub_imgs = c_imgs[s:e].clone().detach().requires_grad_(True)
                sub_ages = c_ages[s:e]
                
                sub_grads = torch.autograd.grad(
                    (wb_model(sub_imgs) - sub_ages).abs().sum(), 
                    list(wb_model.parameters())
                )
                
                grad_flat = torch.cat([g.view(-1) for g in sub_grads])
                clip_f = min(1.0, cfg['dp_max_grad_norm'] / (grad_flat.norm().item() + 1e-8))
                wb_cont[s:e] = torch.dot(cumulative_delta_w, grad_flat * clip_f).item() / len(wb_batches)
            
            wb_scores = wb_cont
            del wb_model
            torch.cuda.empty_cache()

        # --- Evaluation ---
        fair = evaluate_fairness(model, val_loader)
        
        for key in ['overall_mae', 'male_mae', 'female_mae', 'gender_gap', 'race_gap']:
            history[key].append(fair[key])
        
        # FIX: Use RACE_NAMES
        for r in RACE_NAMES:
            history[f'{r}_mae'].append(fair[f'{r}_mae'])
        
        history['train_loss'].append(epoch_loss / epoch_n)
        history['epsilon'].append(privacy_engine.get_epsilon(cfg['dp_delta']))
        history['lambda_t'].append(lambda_t)
        history['target_var'].append(alpha)
        
        if epoch % 5 == 0:
            elapsed = time.time() - start_time
            print(f'[{experiment_name}] Ep {epoch:2d} | L={epoch_loss/epoch_n:.3f} | '
                  f'M={fair["male_mae"]:.2f} F={fair["female_mae"]:.2f} '
                  f'Gap={fair["gender_gap"]:.2f} | ε={history["epsilon"][-1]:.2f} | {elapsed:.0f}s')

    # Final canary losses for black-box scores
    c_imgs, c_ages = canary_ds.get_all_canary_batch()
    with torch.no_grad():
        final_losses = (model(c_imgs.to(device)) - c_ages.to(device)).abs().cpu().numpy()
    
    print(f'[{experiment_name}] Done in {time.time()-start_time:.0f}s')
    
    return {
        'model': model,
        'bb_scores': initial_losses - final_losses,
        'wb_scores': wb_scores,
        'history': history,
        'epsilon_spent': history['epsilon'][-1],
        'final_metrics': fair
    }

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# SAVE RESULTS (FIXED)
# ══════════════════════════════════════════════════════════════════════════════

def convert_to_serializable(obj):
    """Recursively convert numpy types and Path objects to Python native types for JSON serialization."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.int64, np.int32, np.int16, np.int8, np.uint64, np.uint32, np.uint16, np.uint8)):
        return int(obj)
    elif isinstance(obj, (np.float64, np.float32, np.float16)):
        return float(obj)
    elif isinstance(obj, np.bool_):
        return bool(obj)
    elif isinstance(obj, Path):
        return str(obj)
    elif isinstance(obj, dict):
        return {convert_to_serializable(key): convert_to_serializable(value) for key, value in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [convert_to_serializable(item) for item in obj]
    elif isinstance(obj, torch.Tensor):
        return obj.tolist()
    elif isinstance(obj, (np.generic,)):
        return obj.item()
    else:
        return obj

results = {
    'timestamp': datetime.now().isoformat(),
    'algorithm': 'Paper-Accurate Hashimoto Algorithm 1 (Online Mirror Descent)',
    'issues_fixed': [
        '1. White-box: batched canaries + sampled batches',
        '2. Validation leak: canary indices excluded',
        '3. EXACT ALGORITHM 1: Maintains q_t distribution and λ_t dual variable',
        '4. DRO Resampling: Uses WeightedRandomSampler for DP-SGD compatibility',
        '5. Explicit ERM vs ALG1 comparison',
        '6. Race-aware evaluation',
        '7. Dual-GPU execution documented (CUDA_VISIBLE_DEVICES=1)',
        '8. Gradient accumulation option',
        '9. params_before only during WB batches',
    ],
    'config': convert_to_serializable(CFG),
    'erm': {
        'history': convert_to_serializable(res_erm['history']),
        'epsilon_spent': float(res_erm['epsilon_spent']),
        'bb_audit': convert_to_serializable(bb_erm),
        'wb_audit': convert_to_serializable(wb_erm),
        'final_metrics': convert_to_serializable(res_erm['final_metrics']),
    },
    'alg1': {
        'history': convert_to_serializable(res_alg1['history']),
        'epsilon_spent': float(res_alg1['epsilon_spent']),
        'bb_audit': convert_to_serializable(bb_alg1),
        'wb_audit': convert_to_serializable(wb_alg1),
        'final_metrics': convert_to_serializable(res_alg1['final_metrics']),
    },
    'comparison': {
        'gender_gap_reduction': float(res_erm['history']['gender_gap'][-1] - res_alg1['history']['gender_gap'][-1]),
        'gender_gap_reduction_pct': float((res_erm['history']['gender_gap'][-1] - res_alg1['history']['gender_gap'][-1]) / max(res_erm['history']['gender_gap'][-1], 0.01) * 100),
        'race_gap_reduction': float(res_erm['history']['race_gap'][-1] - res_alg1['history']['race_gap'][-1]),
        'race_gap_reduction_pct': float((res_erm['history']['race_gap'][-1] - res_alg1['history']['race_gap'][-1]) / max(res_erm['history']['race_gap'][-1], 0.01) * 100),
        'utility_cost': float(res_alg1['history']['overall_mae'][-1] - res_erm['history']['overall_mae'][-1]),
        'privacy_cost_bb': float(bb_alg1['eps_lb'] - bb_erm['eps_lb']),
        'privacy_cost_wb': float(wb_alg1['eps_lb'] - wb_erm['eps_lb']),
    }
}

out_json = CFG['exp_dir'] / 'paper_accurate_alg1_results.json'
with open(out_json, 'w') as f:
    json.dump(results, f, indent=2)
print(f'Results saved → {out_json}')

# Save both models (only if they exist and are in the results dict)
print('\nSaving models...')
for name in ['erm', 'alg1']:
    result_dict = res_erm if name == 'erm' else res_alg1
    if 'model' in result_dict and result_dict['model'] is not None:
        try:
            ckpt_path = CFG['exp_dir'] / f'{name}_model.pth'
            state_dict = result_dict['model']._module.state_dict() if hasattr(result_dict['model'], '_module') else result_dict['model'].state_dict()
            torch.save(state_dict, ckpt_path)
            print(f'✓ {name.upper()} Model saved → {ckpt_path}')
        except Exception as e:
            print(f'✗ Could not save {name.upper()} model: {e}')
    else:
        print(f'⚠️  {name.upper()} model not found in results (likely deleted to save memory)')

# Also save as pickle for complete Python object preservation
import pickle
pkl_path = CFG['exp_dir'] / 'paper_accurate_alg1_results.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(results, f)
print(f'✓ Results saved as pickle → {pkl_path}')

print(f'\n✓ All results saved to: {CFG["exp_dir"]}')

Results saved → project/experiments/paper_accurate_alg1/paper_accurate_alg1_results.json

Saving models...
⚠️  ERM model not found in results (likely deleted to save memory)
✓ ALG1 Model saved → project/experiments/paper_accurate_alg1/alg1_model.pth
✓ Results saved as pickle → project/experiments/paper_accurate_alg1/paper_accurate_alg1_results.pkl

✓ All results saved to: project/experiments/paper_accurate_alg1


In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION & FINAL REPORT
# ══════════════════════════════════════════════════════════════════════════════
print('\nGenerating Final Report Figure...')
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
epochs_x = range(1, CFG['epochs'] + 1)

# 1. Gender MAE
ax = axes[0, 0]
ax.plot(epochs_x, res_erm['history']['male_mae'], 'b--', label='ERM Male', alpha=0.7)
ax.plot(epochs_x, res_erm['history']['female_mae'], 'r--', label='ERM Female', alpha=0.7)
ax.plot(epochs_x, res_alg1['history']['male_mae'], 'b-', label='ALG1 Male', lw=2)
ax.plot(epochs_x, res_alg1['history']['female_mae'], 'r-', label='ALG1 Female', lw=2)
ax.set_title('Gender MAE: ERM vs Hashimoto ALG1', fontweight='bold')
ax.set_ylabel('MAE')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Gender Gap
ax = axes[0, 1]
ax.plot(epochs_x, res_erm['history']['gender_gap'], 'r--', label='ERM', lw=2)
ax.plot(epochs_x, res_alg1['history']['gender_gap'], 'g-', label='ALG1 (Exact)', lw=2)
ax.set_title('Gender Gap Reduction', fontweight='bold')
ax.set_ylabel('Gap (years)')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Lambda (Dual Variable) tracking
ax = axes[0, 2]
ax.plot(epochs_x, res_alg1['history']['lambda_t'], 'purple', lw=2)
ax.axhline(1.0/(2.0*CFG['dro_alpha']), color='gray', ls='--', label='Max Bound (1/2α)')
ax.set_title('Algorithm 1: Dual Variable λ_t', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('λ')
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Race Gap
ax = axes[1, 0]
ax.plot(epochs_x, res_erm['history']['race_gap'], 'r--', label='ERM', lw=2)
ax.plot(epochs_x, res_alg1['history']['race_gap'], 'g-', label='ALG1', lw=2)
ax.set_title('Race Gap Reduction', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Race Gap')
ax.legend()
ax.grid(True, alpha=0.3)

# 5. BB Scores
ax = axes[1, 1]
m_np = membership.numpy()
ax.hist(res_erm['bb_scores'][m_np==1], bins=25, alpha=0.5, label='ERM IN', color='blue')
ax.hist(res_alg1['bb_scores'][m_np==1], bins=25, alpha=0.6, label='ALG1 IN', color='green', histtype='step', lw=2)
ax.set_title('Black-box MI Scores (IN Canaries)', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# 6. Summary Bar
ax = axes[1, 2]
cats = ['Gender\nGap', 'Race\nGap', 'BB ε_lb', 'WB ε_lb']
erm_vals = [
    res_erm['history']['gender_gap'][-1],
    res_erm['history']['race_gap'][-1],
    bb_erm['eps_lb'],
    wb_erm['eps_lb']
]
alg1_vals = [
    res_alg1['history']['gender_gap'][-1],
    res_alg1['history']['race_gap'][-1],
    bb_alg1['eps_lb'],
    wb_alg1['eps_lb']
]

x = np.arange(len(cats))
width = 0.35
bars1 = ax.bar(x - width/2, erm_vals, width, label='ERM', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, alg1_vals, width, label='ALG1 (Exact)', color='forestgreen', edgecolor='black')
ax.set_title('Key Metrics: ERM vs Hashimoto ALG1', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(cats, fontsize=9)
ax.legend()
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}', xy=(bar.get_x() + bar.get_width()/2, height),
                xytext=(0, 2), textcoords='offset points', ha='center', va='bottom', fontsize=7)

plt.suptitle(
    'Paper-Accurate Hashimoto Algorithm 1 + DP-SGD + Privacy Auditing\n'
    'UTKFace Age Regression (2nd L40s GPU)',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()

fig_path = CFG['exp_dir'] / 'paper_accurate_alg1_comparison.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {fig_path}')


Generating Final Report Figure...
Figure saved → project/experiments/paper_accurate_alg1/paper_accurate_alg1_comparison.png


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# CONSOLIDATED RESULTS TABLE
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 90)
print('CONSOLIDATED RESULTS TABLE')
print('=' * 90)
print(f'{"Metric":<30} {"ERM":>12} {"ALG1 (Exact)":>12} {"Δ":>10} {"%Δ":>8}')
print('-' * 90)

def pct_change(old, new):
    if abs(old) < 0.01:
        return 'N/A'
    return f'{100*(new-old)/old:+.1f}%'

metrics = [
    ('Overall MAE', res_erm['history']['overall_mae'][-1], res_alg1['history']['overall_mae'][-1]),
    ('Male MAE', res_erm['history']['male_mae'][-1], res_alg1['history']['male_mae'][-1]),
    ('Female MAE', res_erm['history']['female_mae'][-1], res_alg1['history']['female_mae'][-1]),
    ('GENDER GAP ↓', res_erm['history']['gender_gap'][-1], res_alg1['history']['gender_gap'][-1]),
    ('RACE GAP ↓', res_erm['history']['race_gap'][-1], res_alg1['history']['race_gap'][-1]),
    ('', '', ''),
    ('Theoretical ε', res_erm['epsilon_spent'], res_alg1['epsilon_spent']),
    ('Black-box ε_lb', bb_erm['eps_lb'], bb_alg1['eps_lb']),
    ('White-box ε_lb', wb_erm['eps_lb'], wb_alg1['eps_lb']),
    ('Black-box AUC', bb_erm['auc'], bb_alg1['auc']),
    ('White-box AUC', wb_erm['auc'], wb_alg1['auc']),
    ('', '', ''),
    ('BB Attack Acc', bb_erm['attack_acc'], bb_alg1['attack_acc']),
    ('WB Attack Acc', wb_erm['attack_acc'], wb_alg1['attack_acc']),
]

for race_name in ['white', 'black', 'asian', 'indian', 'others']:
    metrics.append((
        f'{race_name.capitalize()} MAE',
        res_erm['history'][f'{race_name}_mae'][-1],
        res_alg1['history'][f'{race_name}_mae'][-1]
    ))

for name, erm_val, alg1_val in metrics:
    if name == '':
        print('-' * 90)
        continue
    delta = alg1_val - erm_val
    delta_str = f'{delta:+.3f}' if abs(delta) < 100 else f'{delta:+.1f}'
    pct_str = pct_change(erm_val, alg1_val)
    print(f'{name:<30} {erm_val:>12.3f} {alg1_val:>12.3f} {delta_str:>10} {pct_str:>8}')

print('=' * 90)


CONSOLIDATED RESULTS TABLE
Metric                                  ERM ALG1 (Exact)          Δ       %Δ
------------------------------------------------------------------------------------------
Overall MAE                          12.549       15.954     +3.405   +27.1%
Male MAE                             12.256       16.833     +4.577   +37.3%
Female MAE                           12.830       15.110     +2.281   +17.8%
GENDER GAP ↓                          0.574        1.723     +1.149  +200.3%
RACE GAP ↓                            6.117        7.137     +1.020   +16.7%
------------------------------------------------------------------------------------------
Theoretical ε                         1.054        1.054     +0.000    +0.0%
Black-box ε_lb                        0.000        0.000     +0.000      N/A
White-box ε_lb                        0.000        0.000     +0.000      N/A
Black-box AUC                         0.497        0.547     +0.050   +10.1%
White-box AUC       

In [14]:
# ══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 90)
print('FINAL SUMMARY - PAPER ACCURATE ALGORITHM 1 IMPLEMENTATION')
print('=' * 90)

comp = results['comparison']

print(f"""
KEY FINDINGS:

1.  BIAS REDUCTION (Algorithm 1):
    - Gender gap: {res_erm['history']['gender_gap'][-1]:.3f} → {res_alg1['history']['gender_gap'][-1]:.3f} years ({comp['gender_gap_reduction_pct']:.1f}% reduction)
    - Race gap: {res_erm['history']['race_gap'][-1]:.3f} → {res_alg1['history']['race_gap'][-1]:.3f} years ({comp['race_gap_reduction_pct']:.1f}% reduction)

2.  UTILITY TRADE-OFF:
    - Overall MAE: {res_erm['history']['overall_mae'][-1]:.3f} → {res_alg1['history']['overall_mae'][-1]:.3f} years ({comp['utility_cost']:+.3f})

3.  PRIVACY PRESERVATION:
    - Theoretical ε: ERM={res_erm['epsilon_spent']:.3f}, ALG1={res_alg1['epsilon_spent']:.3f}
    - BB ε_lb: ERM={bb_erm['eps_lb']:.3f}, ALG1={bb_alg1['eps_lb']:.3f}
    - WB ε_lb: ERM={wb_erm['eps_lb']:.3f}, ALG1={wb_alg1['eps_lb']:.3f}

4.  ALGORITHM 1 DETAILS:
    - Uses EXACT Online Mirror Descent with χ²-divergence
    - Maintains q_t distribution and λ_t dual variable
    - DRO Resampling via WeightedRandomSampler for DP-SGD compatibility
    - Final λ_t: {res_alg1['history']['lambda_t'][-1]:.4f}
    - Target variance α: {CFG['dro_alpha']:.2f}

5.  RACE-LEVEL ANALYSIS:
""")

for race_name in ['white', 'black', 'asian', 'indian', 'others']:
    erm_mae = res_erm['history'][f'{race_name}_mae'][-1]
    alg1_mae = res_alg1['history'][f'{race_name}_mae'][-1]
    print(f"    {race_name.capitalize():>8}: ERM={erm_mae:.3f}, ALG1={alg1_mae:.3f}, Δ={alg1_mae-erm_mae:+.3f}")

print('\n' + '=' * 90)
print('EXPERIMENT COMPLETE - Paper-Accurate Algorithm 1 Implementation')
print(f'Results saved to: {CFG["exp_dir"]}')
print('=' * 90)


FINAL SUMMARY - PAPER ACCURATE ALGORITHM 1 IMPLEMENTATION

KEY FINDINGS:

1.  BIAS REDUCTION (Algorithm 1):
    - Gender gap: 0.574 → 1.723 years (-200.3% reduction)
    - Race gap: 6.117 → 7.137 years (-16.7% reduction)

2.  UTILITY TRADE-OFF:
    - Overall MAE: 12.549 → 15.954 years (+3.405)

3.  PRIVACY PRESERVATION:
    - Theoretical ε: ERM=1.054, ALG1=1.054
    - BB ε_lb: ERM=0.000, ALG1=0.000
    - WB ε_lb: ERM=0.000, ALG1=0.000

4.  ALGORITHM 1 DETAILS:
    - Uses EXACT Online Mirror Descent with χ²-divergence
    - Maintains q_t distribution and λ_t dual variable
    - DRO Resampling via WeightedRandomSampler for DP-SGD compatibility
    - Final λ_t: 0.0000
    - Target variance α: 1.00

5.  RACE-LEVEL ANALYSIS:

       White: ERM=15.298, ALG1=18.741, Δ=+3.443
       Black: ERM=11.082, ALG1=11.604, Δ=+0.522
       Asian: ERM=9.679, ALG1=16.122, Δ=+6.442
      Indian: ERM=10.452, ALG1=13.433, Δ=+2.980
      Others: ERM=9.181, ALG1=12.805, Δ=+3.624

EXPERIMENT COMPLETE - Paper-A

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RUN FIXED PAPER-ACCURATE ALGORITHM 1 (DRO SUB-EPOCH SAMPLING)
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 90)
print('RUNNING FIXED PAPER-ACCURATE ALGORITHM 1 (DRO SUB-EPOCH SAMPLING)')
print('=' * 90)
print('''
This implementation uses DRO Sub-epoch sampling to prevent Opacus from 
scaling noise based on the skewed q_t distribution.

Key Fix: 
- Standard uniform DataLoader (shuffle=True) so Opacus assumes uniform probability
- DRO implemented via resampling a subset according to q_t distribution
- Full evaluation on training set for q_t updates
''')

# Clear GPU memory before running
gc.collect()
torch.cuda.empty_cache()

# Run the fixed version
res_fixed = train_model_paper_accurate_fixed(
    cfg=CFG,  # Use the existing config
    train_ds=train_ds, 
    val_loader=val_loader, 
    canary_ds=canary_ds, 
    membership=membership, 
    experiment_name='ALG1_FIXED'
)

# Run audits
bb_fixed = audit_epsilon(res_fixed['bb_scores'], membership)
wb_fixed = audit_epsilon(res_fixed['wb_scores'], membership)

print(f'\n✓ Fixed Algorithm 1 complete!')
print(f'  BB ε_lb: {bb_fixed["eps_lb"]:.4f}')
print(f'  WB ε_lb: {wb_fixed["eps_lb"]:.4f}')
print(f'  Final Gender Gap: {res_fixed["history"]["gender_gap"][-1]:.3f}')
print(f'  Final Race Gap: {res_fixed["history"]["race_gap"][-1]:.3f}')
print(f'  Final λ_t: {res_fixed["history"]["lambda_t"][-1]:.4f}')

# ══════════════════════════════════════════════════════════════════════════════
# COMPARISON TABLE: ORIGINAL vs FIXED
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 90)
print('COMPARISON: ORIGINAL ALG1 vs FIXED ALG1')
print('=' * 90)
print(f'{"Metric":<30} {"Original ALG1":>15} {"Fixed ALG1":>15} {"Δ":>10}')
print('-' * 90)

comparison_metrics = [
    ('Overall MAE', res_alg1['history']['overall_mae'][-1], res_fixed['history']['overall_mae'][-1]),
    ('Male MAE', res_alg1['history']['male_mae'][-1], res_fixed['history']['male_mae'][-1]),
    ('Female MAE', res_alg1['history']['female_mae'][-1], res_fixed['history']['female_mae'][-1]),
    ('Gender Gap ↓', res_alg1['history']['gender_gap'][-1], res_fixed['history']['gender_gap'][-1]),
    ('Race Gap ↓', res_alg1['history']['race_gap'][-1], res_fixed['history']['race_gap'][-1]),
    ('', '', ''),
    ('Theoretical ε', res_alg1['epsilon_spent'], res_fixed['epsilon_spent']),
    ('BB ε_lb', bb_alg1['eps_lb'], bb_fixed['eps_lb']),
    ('WB ε_lb', wb_alg1['eps_lb'], wb_fixed['eps_lb']),
    ('BB Attack Acc', bb_alg1['attack_acc'], bb_fixed['attack_acc']),
    ('WB Attack Acc', wb_alg1['attack_acc'], wb_fixed['attack_acc']),
    ('', '', ''),
    ('Final λ_t', res_alg1['history']['lambda_t'][-1], res_fixed['history']['lambda_t'][-1]),
]

for name, orig_val, fixed_val in comparison_metrics:
    if name == '':
        print('-' * 90)
        continue
    delta = fixed_val - orig_val
    delta_str = f'{delta:+.3f}' if abs(delta) < 100 else f'{delta:+.1f}'
    print(f'{name:<30} {orig_val:>15.3f} {fixed_val:>15.3f} {delta_str:>10}')

print('=' * 90)

# ══════════════════════════════════════════════════════════════════════════════
# SAVE FIXED RESULTS
# ══════════════════════════════════════════════════════════════════════════════

# Save fixed results
fixed_results = {
    'timestamp': datetime.now().isoformat(),
    'algorithm': 'Paper-Accurate Hashimoto Algorithm 1 (Fixed - DRO Sub-epoch)',
    'config': convert_to_serializable(CFG),
    'history': convert_to_serializable(res_fixed['history']),
    'epsilon_spent': float(res_fixed['epsilon_spent']),
    'bb_audit': convert_to_serializable(bb_fixed),
    'wb_audit': convert_to_serializable(wb_fixed),
    'final_metrics': convert_to_serializable(res_fixed['final_metrics']),
    'comparison_with_original': {
        'gender_gap_original': float(res_alg1['history']['gender_gap'][-1]),
        'gender_gap_fixed': float(res_fixed['history']['gender_gap'][-1]),
        'gender_gap_improvement': float(res_alg1['history']['gender_gap'][-1] - res_fixed['history']['gender_gap'][-1]),
        'race_gap_original': float(res_alg1['history']['race_gap'][-1]),
        'race_gap_fixed': float(res_fixed['history']['race_gap'][-1]),
        'race_gap_improvement': float(res_alg1['history']['race_gap'][-1] - res_fixed['history']['race_gap'][-1]),
        'overall_mae_original': float(res_alg1['history']['overall_mae'][-1]),
        'overall_mae_fixed': float(res_fixed['history']['overall_mae'][-1]),
        'lambda_t_final_original': float(res_alg1['history']['lambda_t'][-1]),
        'lambda_t_final_fixed': float(res_fixed['history']['lambda_t'][-1]),
    }
}

fixed_json_path = CFG['exp_dir'] / 'alg1_fixed_results.json'
with open(fixed_json_path, 'w') as f:
    json.dump(fixed_results, f, indent=2)
print(f'\nFixed results saved → {fixed_json_path}')

# Save fixed model if available
if 'model' in res_fixed and res_fixed['model'] is not None:
    fixed_model_path = CFG['exp_dir'] / 'alg1_fixed_model.pth'
    state_dict = res_fixed['model']._module.state_dict() if hasattr(res_fixed['model'], '_module') else res_fixed['model'].state_dict()
    torch.save(state_dict, fixed_model_path)
    print(f'Fixed model saved → {fixed_model_path}')

print('\n' + '=' * 90)
print('✅ FIXED ALGORITHM 1 COMPLETE!')
print('=' * 90)
print(f'''
Summary:
  Original ALG1 λ_t: {res_alg1['history']['lambda_t'][-1]:.4f}
  Fixed ALG1 λ_t:    {res_fixed['history']['lambda_t'][-1]:.4f}
  
  Original Gender Gap: {res_alg1['history']['gender_gap'][-1]:.3f}
  Fixed Gender Gap:    {res_fixed['history']['gender_gap'][-1]:.3f}
  
  Original Overall MAE: {res_alg1['history']['overall_mae'][-1]:.3f}
  Fixed Overall MAE:    {res_fixed['history']['overall_mae'][-1]:.3f}
''')
print('=' * 90)